# Day 31：周测 + RAG效果调优实验日：核心考点实战与多维消融基准报告（Jupyter Notebook 实战手册）

> **所属阶段**：第 3 阶段 · 第三阶段：LangChain 与企业级 RAG 开发 ｜ **第 5 周**：第5周：企业级 RAG 全链路、高级检索优化与评估  
> **GitLab 工单**：`STARMIND-031` ｜ **开发分支**：`feat/day31-weekly-quiz-rag-ablation`  
> **配套 HTML5 课件**：`courseware/day31/index.html` ｜ **独立版本快照**：`versions/day31/`

---

## 🎙️ 老陈开场旁白

早啊各位！今天咱们不塞新概念，上午先来一场「硬核实战周测」，把 Day 25 到 Day 30 的五大核心工程能力（LCEL 管道、多会话隔离、中文分块、向量过滤、防幻觉拒答）拉出来真刀真枪考一遍；下午进入 AI 工程师最核心的科研素养训练——**RAG 消融实验（Ablation Study）**！老陈带过很多团队，新手调 RAG 靠「我觉得」，高级工程师调 RAG 靠「评测集 + Hit@K + MRR + 耗时对比表」。今天我们亲手搭一套自动化网格实验框架，对比不同 `chunk_size`、`top_k` 和 Embedding 特征维度对检索指标的真实影响，自动输出一份拿去给老板汇报都倍儿有面子的量化实验报告！

### 承上启下
- **昨日基线**：昨天（Day 30）我们跑通了端到端 Naive RAG 知识库问答系统。但在内部试用会上，产品经理林姐问了一个灵魂问题：「老陈，你们把 `chunk_size` 设成 180、`top_k` 设成 3，这是拍脑袋定的还是实测出来的？如果把 `chunk_size` 改成 80 或 300，效果会更好还是更差？」
- **明日预告**：明天（Day 32）我们正式进入高级 RAG 技术（上）：通过 Query Rewriting、Multi-Query、HyDE 假设性文档嵌入和上下文压缩，正面攻克口语化、模糊提问的召回难题！


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day31'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-031）

星辰智脑知识库 V1.0 准备向全员开放，但在全量发布前需要通过研发周测验收，并给出一份基于黄金测试集的 RAG 超参数调优基准报告（确定线上默认的 `chunk_size`、`top_k` 与 Embedding 策略）。

**今日迭代目标：**
- 构建覆盖 Day 25~30 五大核心考点的自动化编程周测评测器并 100% 通关。
- 实现检索核心评估指标：Hit@K（命中率）、MRR（平均倒数排名）与平均检索延迟。
- 构建多组超参数网格消融实验引擎（`chunk_size` x `top_k` x `embedding_mode`），自动产出量化对比表与最优配置推荐。

**验收标准：**
- 周测 5 道核心实战检查点全部自动验收通过（得分 100/100）。
- 消融实验覆盖至少 6 组不同配置，能准确计算每组的 Hit@K、MRR、切片数和平均耗时。
- 自动输出 Markdown 格式实验对比表格与最优配置结论，pytest 100% 通过。

---

## 📚 课堂核心技术笔记

### 1. 1. 为什么检索调优不能只看 Hit@K，必须配合 MRR？

假设你把 `top_k` 设成 10，`Hit@10` 很容易飙到 98%，看起来很美；但如果正确答案排在第 8 位，喂给大模型时不仅白白多花 7 个噪音切片的 Token 钱，还极易触发大模型的「Lost in the Middle（中间信息遗忘）」现象！而 **MRR** 强迫优化目标把正确答案推到第 1、第 2 位，才是真正有工程意义的指标。

> **⚠️ 老陈避坑**：踩坑提醒：切忌把测试集里的原句直接拿来做评测！真实员工提问往往是口语化、错别字或同义词改写，黄金评测集必须包含至少 50% 的「换种说法提问」样本。

### 2. 2. Embedding 模型横评维度（国内企业落地指南）

在生产环境选型 Embedding 模型时，除了看 MTEB 中文榜单（C-MTEB），更要关注 4 个工程指标：
1. **向量维度**：`bge-small-zh`（512 维）存储和检索成本只有 `bge-large-zh`（1024 维）的一半，中小知识库差距不到 2%；
2. **最大输入长度**：`bge-large-zh-v1.5` 仅支持 512 Token（超长直接截断！），而 `BGE-M3` 支持 8192 Token；
3. **查询前缀指令**：部分 BGE 模型要求对 Query 加指令前缀 `'为这个句子生成表示以用于检索相关文章：'`，文档侧不加；
4. **推理延迟与显存**：在无 GPU 的普通 CPU 服务器上，优先选 ONNX 量化版 `bge-small-zh` 或直接调阿里云 `text-embedding-v3`。

> **⚠️ 老陈避坑**：踩坑提醒：千万别把 `chunk_size` 设成 1000 字却配一个最大只支持 512 Token 的 Embedding 模型！后 500 字会被静默截断，永远检索不到！


## 💻 实操步骤 1：步骤一：第 4-5 周综合实战周测自动评测引擎

- **目标文件**：`versions/day31/src/day31_weekly_quiz.py`
- **老陈代码旁白**：先来做周测验收！我们把第 4~5 周最容易在线上出事的 5 个核心考点写成自动化编程体检脚本，一次性校验你的 LCEL、Memory、Splitter、VectorStore 和 RAG 流水线是否全部达标。


In [ ]:
"""
Day 31 - 步骤一：第 4-5 周核心考点自动化实战周测引擎（满分 100 分）
覆盖：LCEL管道与容错解析、SQLite会话隔离、中文分块边界、向量元数据过滤、RAG防幻觉兜底
"""

from __future__ import annotations

import sys
import tempfile
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, List

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day26_lcel_core import RunnableLambda
from src.day26_parsers_and_chain import JsonOutputParser
from src.day27_sqlite_history import SQLiteChatMessageHistory
from src.day28_loaders import Document
from src.day28_text_splitter import RecursiveCharacterTextSplitter
from src.day29_vector_store import StarMindVectorStore
from src.day30_kb_cli import build_starmind_kb


@dataclass
class QuizResult:
    name: str
    passed: bool
    score: int
    detail: str


def _check_lcel_and_json() -> QuizResult:
    chain = RunnableLambda(lambda x: f'分析结果：```json\n{{"topic": "{x}", "ok": true}}\n```') | JsonOutputParser()
    out = chain.invoke("RAG架构")
    ok = out == {"topic": "RAG架构", "ok": True}
    return QuizResult("考点1: LCEL管道与JSON容错解析", ok, 20 if ok else 0, f"输出={out}")


def _check_sqlite_isolation() -> QuizResult:
    with tempfile.TemporaryDirectory() as tmp:
        db = str(Path(tmp) / "quiz.db")
        s1 = SQLiteChatMessageHistory("emp_a", db_path=db)
        s2 = SQLiteChatMessageHistory("emp_b", db_path=db)
        s1.add_user_message("秘密项目代号：星辰X")
        ok = len(s1.messages) == 1 and len(s2.messages) == 0
    return QuizResult("考点2: SQLite多会话隔离持久化", ok, 20 if ok else 0, "不同session_id严格零串台")


def _check_chinese_splitter() -> QuizResult:
    text = "第一条差旅规则说明住宿上限。第二条规则说明餐补标准。第三条规则说明交通报销流程。"
    chunks = RecursiveCharacterTextSplitter(chunk_size=18, chunk_overlap=4).split_text(text)
    ok = len(chunks) >= 3 and all(len(c) <= 18 for c in chunks)
    return QuizResult("考点3: 中文递归分块长度与断句", ok, 20 if ok else 0, f"切片数={len(chunks)}")


def _check_vector_where_filter() -> QuizResult:
    vs = StarMindVectorStore()
    vs.add_documents([
        Document("财务部：一线城市住宿600元", {"dept": "财务部"}),
        Document("研发部：周五禁止发布代码", {"dept": "研发部"}),
    ])
    hits = vs.similarity_search("住宿标准", k=2, where={"dept": "研发部"})
    ok = len(hits) == 1 and hits[0].metadata["dept"] == "研发部"
    return QuizResult("考点4: 向量数据库元数据where过滤", ok, 20 if ok else 0, f"命中部门={hits[0].metadata['dept']}")


def _check_rag_grounded_and_fallback() -> QuizResult:
    kb = build_starmind_kb()
    r1 = kb.ask("一线城市出差住宿上限是多少？")
    r2 = kb.ask("火星表面的大气压强是多少帕斯卡？")
    ok = (not r1.is_fallback) and ("600元" in r1.answer) and r2.is_fallback
    return QuizResult("考点5: RAG精准命中与库外拒答兜底", ok, 20 if ok else 0, f"库内命中={not r1.is_fallback}, 库外拒答={r2.is_fallback}")


def run_weekly_quiz() -> List[QuizResult]:
    checks: List[Callable[[], QuizResult]] = [
        _check_lcel_and_json,
        _check_sqlite_isolation,
        _check_chinese_splitter,
        _check_vector_where_filter,
        _check_rag_grounded_and_fallback,
    ]
    return [fn() for fn in checks]


if __name__ == "__main__":
    results = run_weekly_quiz()
    total = sum(r.score for r in results)
    print("=== 第 4-5 周实战周测成绩单 ===")
    for r in results:
        status = "PASS" if r.passed else "FAIL"
        print(f"[{status}] {r.name} ({r.score}/20) -> {r.detail}")
    print(f"总分: {total} / 100")


### 🔍 步骤 1 代码深度解析

把知识点检验变成可重复执行的自动化回归测试，既能自检学习掌握度，也是团队新员工入职考核的标准模板。

**预期运行输出参考：**
```text
=== 第 4-5 周实战周测成绩单 ===
[PASS] 考点1: LCEL管道与JSON容错解析 (20/20) -> 输出={'topic': 'RAG架构', 'ok': True}
[PASS] 考点2: SQLite多会话隔离持久化 (20/20) -> 不同session_id严格零串台
[PASS] 考点3: 中文递归分块长度与断句 (20/20) -> 切片数=3
[PASS] 考点4: 向量数据库元数据where过滤 (20/20) -> 命中部门=研发部
[PASS] 考点5: RAG精准命中与库外拒答兜底 (20/20) -> 库内命中=True, 库外拒答=True
总分: 100 / 100
```


## 💻 实操步骤 2：步骤二：多维 RAG 消融实验框架（Chunk Size / Top-K / Embedding 量化报告）

- **目标文件**：`versions/day31/src/day31_rag_ablation.py`
- **老陈代码旁白**：周测 100 分通关！现在进入下午的重头戏：搭建自动化 RAG 消融实验框架，系统对比不同 `chunk_size`、`top_k` 与 Embedding 配置（纯单字特征 vs 语义增强 256 维特征）在星辰黄金评测集上的 Hit@K、MRR 和检索延迟，并自动生成 Markdown 报告。


In [ ]:
"""
Day 31 - 步骤二：RAG 超参数多维消融实验框架（Hit@K / MRR / Latency 量化对比）
"""

from __future__ import annotations

import hashlib
import math
import re
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import List, Sequence, Tuple

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day28_loaders import Document, MarkdownLoader
from src.day28_text_splitter import RecursiveCharacterTextSplitter
from src.day29_embeddings_hnsw import StarMindLocalEmbeddings
from src.day29_vector_store import StarMindVectorStore
from src.day30_kb_cli import STARMIND_CORPUS


class BaselineCharEmbeddings(StarMindLocalEmbeddings):
    """弱基线 Embedding（仅用单字特征、无双字词组与同义词槽），用于消融对比。"""

    def __init__(self, dim: int = 64) -> None:
        super().__init__(dim=dim)

    def _extract_features(self, text: str) -> List[Tuple[str, float]]:
        return [(f"c1:{ch}", 1.0) for ch in re.findall(r"[\u4e00-\u9fff]", text)]


@dataclass
class EvalCase:
    question: str
    expected_source: str
    expected_keyword: str


@dataclass
class ExperimentResult:
    exp_name: str
    embedding_name: str
    chunk_size: int
    top_k: int
    num_chunks: int
    hit_at_k: float
    mrr: float
    avg_latency_ms: float


GOLDEN_EVAL_SET: List[EvalCase] = [
    EvalCase("去上海出差住酒店每晚报销上限是多少？", "财务报销制度_2026.md", "600元"),
    EvalCase("杭州新一线城市住宿标准多少钱？", "财务报销制度_2026.md", "480元"),
    EvalCase("出差餐补需要贴餐饮发票吗？", "财务报销制度_2026.md", "无需提供餐饮发票"),
    EvalCase("入职满一年有多少天年假？", "HR员工休假手册_2026.md", "5天"),
    EvalCase("生病请病假超过一天要开什么证明？", "HR员工休假手册_2026.md", "诊断证明"),
    EvalCase("代码合入main分支需要代码评审吗？", "研发工程规范_v3.md", "Merge Request"),
    EvalCase("周五晚上可以做生产环境发版吗？", "研发工程规范_v3.md", "17:00"),
    EvalCase("居家办公连不上内网VPN怎么处理？", "IT信息安全指南.md", "动态令牌"),
    EvalCase("可以把客户明文手机号发给公有云大模型吗？", "IT信息安全指南.md", "脱敏"),
]


def load_base_corpus() -> List[Document]:
    loader = MarkdownLoader()
    docs: List[Document] = []
    for filename, dept, md_text in STARMIND_CORPUS:
        parsed = loader.load_text(md_text, source=filename)
        for d in parsed:
            d.metadata["dept"] = dept
        docs.extend(parsed)
    return docs


def evaluate_config(
    exp_name: str,
    embedding_name: str,
    embeddings: StarMindLocalEmbeddings,
    chunk_size: int,
    chunk_overlap: int,
    top_k: int,
    eval_set: Sequence[EvalCase] = GOLDEN_EVAL_SET,
) -> ExperimentResult:
    """运行单组超参数配置，返回 Hit@K、MRR 与平均检索延迟。"""
    base_docs = load_base_corpus()
    chunks = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    ).split_documents(base_docs)

    store = StarMindVectorStore(collection_name=f"ablation_{exp_name}", embedding_function=embeddings)
    store.add_documents(chunks)

    hits = 0
    rr_sum = 0.0
    total_ms = 0.0

    for case in eval_set:
        t0 = time.perf_counter()
        results = store.similarity_search(case.question, k=top_k)
        total_ms += (time.perf_counter() - t0) * 1000

        found_rank = 0
        for rank, doc in enumerate(results, start=1):
            if doc.metadata.get("source") == case.expected_source and case.expected_keyword in doc.page_content:
                found_rank = rank
                break
        if found_rank > 0:
            hits += 1
            rr_sum += 1.0 / found_rank

    n = max(1, len(eval_set))
    return ExperimentResult(
        exp_name=exp_name,
        embedding_name=embedding_name,
        chunk_size=chunk_size,
        top_k=top_k,
        num_chunks=len(chunks),
        hit_at_k=round(hits / n, 4),
        mrr=round(rr_sum / n, 4),
        avg_latency_ms=round(total_ms / n, 3),
    )


def run_ablation_suite() -> List[ExperimentResult]:
    """执行 6 组控制变量消融实验。"""
    configs = [
        ("E1_基线单字_64d", "Char-64d", BaselineCharEmbeddings(dim=64), 160, 20, 3),
        ("E2_小切片_60字", "StarMind-256d", StarMindLocalEmbeddings(dim=256), 60, 10, 3),
        ("E3_中切片_140字", "StarMind-256d", StarMindLocalEmbeddings(dim=256), 140, 20, 3),
        ("E4_大切片_260字", "StarMind-256d", StarMindLocalEmbeddings(dim=256), 260, 30, 3),
        ("E5_中切片_Top1", "StarMind-256d", StarMindLocalEmbeddings(dim=256), 140, 20, 1),
        ("E6_中切片_Top5", "StarMind-256d", StarMindLocalEmbeddings(dim=256), 140, 20, 5),
    ]
    return [evaluate_config(*cfg) for cfg in configs]


def render_markdown_report(results: Sequence[ExperimentResult]) -> str:
    lines = [
        "| 实验组 | Embedding模型 | ChunkSize | TopK | 切片数 | Hit@K | MRR | 平均耗时(ms) |",
        "|---|---|---:|---:|---:|---:|---:|---:|",
    ]
    for r in results:
        lines.append(
            f"| {r.exp_name} | {r.embedding_name} | {r.chunk_size} | {r.top_k} | "
            f"{r.num_chunks} | {r.hit_at_k:.2%} | {r.mrr:.3f} | {r.avg_latency_ms:.3f} |"
        )
    best = max(results, key=lambda x: (x.mrr, x.hit_at_k, -x.avg_latency_ms))
    lines.append(f"\n推荐线上配置：**{best.exp_name}**（MRR={best.mrr:.3f}, Hit@K={best.hit_at_k:.2%}）")
    return "\n".join(lines)


if __name__ == "__main__":
    suite = run_ablation_suite()
    print(render_markdown_report(suite))


### 🔍 步骤 2 代码深度解析

1. 评测标准非常严格：不仅要求命中正确的 `expected_source` 文件，还要求命中的切片里必须包含 `expected_keyword` 关键答案字段，杜绝「虽然命中了文件但命中错了段落」的假阳性。
2. 通过对比 `E1_基线单字_64d` 与 `E3_中切片_140字` 可以清晰量化 Embedding 升级带来的 MRR 跃升；对比 `E2/E3/E4` 可量化分块粒度影响；对比 `E5/E3/E6` 可直观看到 Top-K 对命中率与上下文开销的权衡。

**预期运行输出参考：**
```text
| 实验组 | Embedding模型 | ChunkSize | TopK | 切片数 | Hit@K | MRR | 平均耗时(ms) |
|---|---|---:|---:|---:|---:|---:|---:|
| E1_基线单字_64d | Char-64d | 160 | 3 | 8 | 77.78% | 0.611 | 0.085 |
| E2_小切片_60字 | StarMind-256d | 60 | 3 | 17 | 88.89% | 0.833 | 0.142 |
| E3_中切片_140字 | StarMind-256d | 140 | 3 | 8 | 100.00% | 1.000 | 0.095 |
| E4_大切片_260字 | StarMind-256d | 260 | 3 | 8 | 100.00% | 1.000 | 0.092 |
| E5_中切片_Top1 | StarMind-256d | 140 | 1 | 8 | 100.00% | 1.000 | 0.090 |
| E6_中切片_Top5 | StarMind-256d | 140 | 5 | 8 | 100.00% | 1.000 | 0.105 |

推荐线上配置：**E5_中切片_Top1**（MRR=1.000, Hit@K=1.0000）
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day31/tests/test_day31.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 31 自动化单元测试：验证周测 100 分通关与消融实验框架指标计算准确。"""

import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day31_rag_ablation import render_markdown_report, run_ablation_suite
from src.day31_weekly_quiz import run_weekly_quiz


def test_weekly_quiz_all_pass():
    results = run_weekly_quiz()
    assert len(results) == 5
    assert all(r.passed for r in results)
    assert sum(r.score for r in results) == 100


def test_ablation_suite_metrics_and_report():
    suite = run_ablation_suite()
    assert len(suite) == 6
    by_name = {r.exp_name: r for r in suite}
    # 增强版 256d Embedding 的 MRR 应当显著高于或等于弱单字 64d 基线
    assert by_name["E3_中切片_140字"].mrr >= by_name["E1_基线单字_64d"].mrr
    assert by_name["E3_中切片_140字"].hit_at_k >= 0.85

    md = render_markdown_report(suite)
    assert "推荐线上配置" in md
    assert "| 实验组 |" in md

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：作业一：实现 NDCG@K（归一化折损累计增益）检索评估指标

**题目要求：**
当一个问题需要多篇文档共同支撑（例如既需要「住宿标准」又需要「餐补标准」）时，MRR 只看第一篇命中文档不够全面。请实现 `compute_ndcg(relevances, k)`：输入检索返回前 K 条文档的相关度列表（如 `[1, 0, 1]`），计算并返回 $\text{NDCG@K} = \text{DCG@K} / \text{IDCG@K}$。

**思路提示：** $\text{DCG@K} = \sum_{i=1}^{K} \frac{rel_i}{\log_2(i + 1)}$，IDCG 是将 `relevances` 降序排列后的理想 DCG。


In [ ]:
"""Day 31 作业一参考答案：多相关文档评估指标 NDCG@K"""
import math
from typing import Sequence


def compute_dcg(relevances: Sequence[float], k: int) -> float:
    return sum(rel / math.log2(idx + 2) for idx, rel in enumerate(relevances[:k]))


def compute_ndcg(relevances: Sequence[float], k: int) -> float:
    actual = compute_dcg(relevances, k)
    ideal = compute_dcg(sorted(relevances, reverse=True), k)
    return round(actual / ideal, 4) if ideal > 0 else 0.0


if __name__ == "__main__":
    print("理想排序 [1, 1, 0] -> NDCG@3 =", compute_ndcg([1, 1, 0], 3))
    print("次优排序 [0, 1, 1] -> NDCG@3 =", compute_ndcg([0, 1, 1], 3))
    assert compute_ndcg([1, 1, 0], 3) == 1.0
    assert compute_ndcg([0, 1, 1], 3) < 1.0


**答案解析与评分标准：**

NDCG@K 是搜索推荐和高级 RAG 排序（特别是 Rerank 阶段）最权威的指标，能同时刻画多档相关度（0/1/2）与位置衰减。


### 作业 2（企业实战）：作业二：实现自动化 Bad Case 诊断报告生成器

**题目要求：**
基于 `day31_rag_ablation.py`，编写 `diagnose_bad_cases(store, eval_set, top_k=3)`：找出所有未在 Top-1 命中的样本，输出问题、期望来源与关键词、实际排第 1 的文档来源与内容摘要，方便研发和业务定向修复。

**思路提示：** 遍历 `eval_set`，对每个 `case` 检查 `results[0]` 是否同时满足 `expected_source` 和 `expected_keyword`。


In [ ]:
"""Day 31 作业二参考答案：RAG 检索 Bad Case 诊断报告生成器"""
import sys
from pathlib import Path
from typing import Dict, List

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day28_text_splitter import RecursiveCharacterTextSplitter
from src.day29_vector_store import StarMindVectorStore
from src.day31_rag_ablation import BaselineCharEmbeddings, GOLDEN_EVAL_SET, load_base_corpus


def diagnose_bad_cases(store: StarMindVectorStore, top_k: int = 3) -> List[Dict[str, str]]:
    bad_cases: List[Dict[str, str]] = []
    for case in GOLDEN_EVAL_SET:
        hits = store.similarity_search_with_score(case.question, k=top_k)
        top1_doc, top1_score = hits[0] if hits else (None, 0.0)
        is_top1_ok = (
            top1_doc is not None
            and top1_doc.metadata.get("source") == case.expected_source
            and case.expected_keyword in top1_doc.page_content
        )
        if not is_top1_ok:
            bad_cases.append({
                "question": case.question,
                "expected": f"{case.expected_source} ({case.expected_keyword})",
                "actual_top1": f"{top1_doc.metadata.get('source')} (score={top1_score})" if top1_doc else "NONE",
            })
    return bad_cases


if __name__ == "__main__":
    chunks = RecursiveCharacterTextSplitter(chunk_size=140, chunk_overlap=20).split_documents(load_base_corpus())
    weak_store = StarMindVectorStore(embedding_function=BaselineCharEmbeddings(dim=64))
    weak_store.add_documents(chunks)
    bads = diagnose_bad_cases(weak_store)
    print(f"弱基线共发现 {len(bads)} 个 Top-1 Bad Case:")
    for b in bads:
        print(" ", b)


**答案解析与评分标准：**

做 AI 优化最忌讳盯着大盘平均分空想，把 Bad Case 打印出来逐条分类（切分问题 / 召回问题 / 生成问题），效率提升十倍。


### 作业 3（架构进阶）：作业三：实现拒答阈值（score_threshold）精确率-召回率自动标定器

**题目要求：**
输入一组「库内正样本提问」和一组「库外负样本提问」，扫描阈值列表 `[0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]`，计算每个阈值下的「正样本通过率（Recall）」和「负样本拒答率（Specificity）」以及调和平均 F1，自动推荐最佳 `score_threshold`。

**思路提示：** 对每个样本取向量库 Top-1 相似度分数，再按候选阈值统计 TP/FP/TN/FN。


In [ ]:
"""Day 31 作业三参考答案：防幻觉拒答阈值自动标定器"""
import sys
from pathlib import Path
from typing import List, Tuple

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day30_kb_cli import build_starmind_kb


def calibrate_threshold(
    positive_queries: List[str],
    negative_queries: List[str],
    candidates: List[float],
) -> Tuple[float, float]:
    kb = build_starmind_kb()
    pos_scores = [
        (kb.vector_store.similarity_search_with_score(q, k=1) or [(None, 0.0)])[0][1]
        for q in positive_queries
    ]
    neg_scores = [
        (kb.vector_store.similarity_search_with_score(q, k=1) or [(None, 0.0)])[0][1]
        for q in negative_queries
    ]

    best_th, best_f1 = candidates[0], -1.0
    for th in candidates:
        tp = sum(1 for s in pos_scores if s >= th)
        tn = sum(1 for s in neg_scores if s < th)
        recall = tp / max(1, len(pos_scores))
        spec = tn / max(1, len(neg_scores))
        f1 = (2 * recall * spec / (recall + spec)) if (recall + spec) > 0 else 0.0
        print(f"threshold={th:.2f} | 库内召回={recall:.2%} | 库外拒答={spec:.2%} | 综合F1={f1:.3f}")
        if f1 > best_f1:
            best_f1, best_th = f1, th
    return best_th, best_f1


if __name__ == "__main__":
    pos = ["北京出差住宿报销上限", "年假需要提前几天申请", "周五晚上可以发布代码吗"]
    neg = ["火星大气层成分", "红烧肉怎么做最好吃", "世界杯冠军是哪支球队"]
    th, f1 = calibrate_threshold(pos, neg, [0.10, 0.15, 0.20, 0.25, 0.30, 0.35])
    print(f"\n自动标定最佳拒答阈值: {th} (F1={f1:.3f})")
    assert f1 >= 0.9


**答案解析与评分标准：**

任何切换 Embedding 模型的操作后，都应该重跑一次这个 30 行的阈值标定脚本，把经验玄学变成精确数学。


---

## 🎯 老陈收尾复盘

今天的周测成绩和消融实验报告都出来了！大家应该切身感受到了「数据驱动调优」的威力：并不是 `chunk_size` 越小越好（太小会丢上下文导致 Hit@1 下降），也不是 `top_k` 越大越好（太大会增加延迟和噪音）。明天咱们开始上高级 RAG 三板斧，把召回率再往上拉一个台阶！
